# Kaggle keeps at most 500 output files: check every map exists, zip each map folder, delete the cloned code (182 files)
import json, glob, os, zipfile
import pandas as pd
for f in sorted(glob.glob(f'{OUT}/*_summary.json')):
    print(f, json.dumps(json.load(open(f)), indent=1))
pred_dirs = sorted(glob.glob(f'{OUT}/*_pred'))
assert pred_dirs, f'no map folders in {OUT}'
for d in pred_dirs:
    split = os.path.basename(d)[:-len('_pred')]
    expected = len(pd.read_csv(f'{OUT}/{split}_metrics.csv'))
    n = len(glob.glob(d + '/*.npz'))
    print(f'{d}: {n} maps (expected {expected})')
    assert n == expected, f'{split}: {n} maps but {expected} cases evaluated -- maps missing, do not save this run'
    sh(f'cd {OUT} && zip -q -r {split}_pred.zip {split}_pred')
    with zipfile.ZipFile(f'{OUT}/{split}_pred.zip') as z:
        nz = sum(name.endswith('.npz') for name in z.namelist())
    assert nz == expected, f'{split}_pred.zip holds {nz} maps, expected {expected}'
    sh(f'rm -rf {d}')
os.chdir('/kaggle/working')
sh(f'rm -rf {CODE_DIR} /kaggle/working/.virtual_documents')
n_files = sum(len(fs) for _, _, fs in os.walk('/kaggle/working'))
print('output files:', n_files)
assert n_files < 500, f'{n_files} output files -- Kaggle keeps only 500'
sh(f'ls -la {OUT} && du -sh {OUT}')
print('ALL MAPS SAVED')


In [ ]:
FOLD = 0                      # 0, 1, 2  or  'full'

REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
SEG_CKPT   = '/kaggle/input/datasets/kumarvemuri1/cids-checkpoints/segresnet_img_e100_best.pt'   # FOLD='full' only
EPOCHS        = 100
TIME_BUDGET_H = 10.0          # training stops before this; the held-out maps are still saved (~30 min)
RESUME_FROM   = ''            # last.pt of an unfinished run of the SAME fold (attached output), to continue
RECOVER_CKPT  = ''            # best.pt of a FINISHED fold whose maps were lost: no training, only re-predicts the held-out maps (~45 min)
CODE_DIR = '/kaggle/working/cids'
OUT      = f'/kaggle/working/seg_fold{FOLD}'
PATHS = [CACHE_ROOT] + ([SEG_CKPT] if FOLD == 'full' else [])


In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not).
    Progress bars redraw a line with '\r'; only each line's final state is printed (one line per epoch)."""
    import subprocess
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf = b''
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        buf += chunk
        *lines, buf = buf.split(b'\n')
        for line in lines:
            print(line.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if buf.strip():
        print(buf.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')


In [ ]:
sh(f'pip install -q monai medim einops appdirs')
sh(f'python -m pytest -q tests')

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
for q in PATHS:
    print('OK ' if Path(q).exists() else '!! MISSING', q)
assert all(Path(q).exists() for q in PATHS)
if RECOVER_CKPT:
    assert FOLD != 'full' and Path(RECOVER_CKPT).exists(), f'RECOVER_CKPT not found: {RECOVER_CKPT}'
    print('recovery mode: re-predicting held-out maps of fold', FOLD, 'from', RECOVER_CKPT)
assert FOLD in (0, 1, 2, 'full'), FOLD
import pandas as pd
sp = pd.read_csv('configs/splits.csv')
if FOLD != 'full':
    print(sp[f'fold3_{FOLD}'].value_counts().to_dict())


## Smoke test (~3 min): 6 training cases, 1 epoch, 3 cases evaluated


In [ ]:
if FOLD == 'full':
    sh(f'mkdir -p /kaggle/working/seg_smoke && cp {SEG_CKPT} /kaggle/working/seg_smoke/best.pt')
    sh(f'python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --out /kaggle/working/seg_smoke --eval_only --eval_split val,test --save_prob --limit_eval 3')
else:
    sh(f'python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --out /kaggle/working/seg_smoke --split_col fold3_{FOLD} --eval_split heldout --save_prob --epochs 1 --val_every 1 --limit_train 6 --limit_eval 3')
sh('ls /kaggle/working/seg_smoke && rm -rf /kaggle/working/seg_smoke')


## Full run
**Read:** for a fold, the val Dice every 5 epochs (the full model reached ~0.72 at 100 epochs; a fold model on 2/3 of the data should land a little lower), then `HELDOUT (~360 cases)`. That is the honest Dice on patients this model never saw. For `'full'`: `VAL` and `TEST` should be close to notebook 02's raw-mask numbers (test raw Dice was ~0.64–0.67; our reported 0.684 is after rel20 post-processing).


In [ ]:
if FOLD == 'full':
    sh(f'mkdir -p {OUT} && cp {SEG_CKPT} {OUT}/best.pt')
    sh(f'python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --out {OUT} --eval_only --eval_split val,test --save_prob')
elif RECOVER_CKPT:
    sh(f'mkdir -p {OUT} && cp {RECOVER_CKPT} {OUT}/best.pt')
    sh(f'python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --out {OUT} --split_col fold3_{FOLD} --eval_split heldout --save_prob --eval_only')
else:
    resume = f'--resume_from {RESUME_FROM}' if RESUME_FROM else ''
    sh(f'python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --out {OUT} --split_col fold3_{FOLD} --eval_split heldout --save_prob --epochs {EPOCHS} --time_budget_h {TIME_BUDGET_H} {resume}')


In [ ]:
# Kaggle keeps at most 500 output files: zip each map folder and delete the cloned code (182 files)
import json, glob, os
for f in sorted(glob.glob(f'{OUT}/*_summary.json')):
    print(f, json.dumps(json.load(open(f)), indent=1))
for d in sorted(glob.glob(f'{OUT}/*_pred')):
    n = len(glob.glob(d + '/*.npz'))
    print(d, n, 'maps')
    sh(f'cd {OUT} && zip -q -r {os.path.basename(d)}.zip {os.path.basename(d)} && rm -rf {os.path.basename(d)}')
os.chdir('/kaggle/working')
sh(f'rm -rf {CODE_DIR} /kaggle/working/.virtual_documents')
sh(f'ls -la {OUT} && du -sh {OUT} && find /kaggle/working -type f | wc -l')
